### Prepare the PCA dataset

Prepare the flow measurements for PCA and keep their labels and timestamps for Section 2 of the assignment.


In [24]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC

from src.preprocessing import DropAllZeroColumns, SpearmanFilter

In [2]:
df = pd.read_csv("dataset.csv")
df.columns = df.columns.str.strip()
flows = df.set_index("Unnamed: 0").drop(
    columns=[
        "Flow ID",
        "Source IP",
        "Source Port",
        "Destination IP",
        "Destination Port",
    ]
)

/var/folders/bx/x30pqcq14gqbqsl_ywfxy6qc0000gn/T/ipykernel_64062/108652544.py:1: DtypeWarning: Columns (85: SimillarHTTP) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dataset.csv")


### Choose the PCA inputs

The original row ID keeps each flow aligned with its label and timestamp. `Flow ID` duplicates information in other columns. The prepared dataset also omits the source and destination IP addresses and ports.

PCA uses measurements and counts. `Inbound` and protocol indicators such as `Protocol_UDP` represent categories and stay with the metadata for inspection.


In [3]:
# Protocol numbers from the IANA registry.
# https://www.iana.org/assignments/protocol-numbers/protocol-numbers.xhtml
protocol_names = {0: "HOPOPT", 6: "TCP", 17: "UDP"}
flows["Protocol"] = (
    flows["Protocol"]
    .map(protocol_names)
    .fillna("Unknown_" + flows["Protocol"].astype(str))
)
flows = pd.get_dummies(flows, columns=["Protocol"], dtype=int)

categorical_columns = {"label", "Inbound"}
measurement_columns = [
    column
    for column in flows.select_dtypes(include="number").columns
    if column not in categorical_columns and not column.startswith("Protocol_")
]
X = flows[measurement_columns]
metadata = flows.drop(columns=measurement_columns)
y = metadata["label"].copy()

### Define the preprocessing pipeline

1. Drop measurements that are zero in every row used to fit the pipeline.
2. Scan the remaining measurements in column order. Drop a measurement if its absolute Spearman correlation with any retained measurement exceeds 0.9. For example, if two columns have a correlation of -0.95, keep the first.
3. Standardize the retained measurements so their units do not dominate PCA.
4. Keep enough principal components to explain at least 99% of the variance in the standardized measurements.
5. Name the component columns `PC1`, `PC2`, and so on. Pandas output carries the original row IDs through scaling and PCA.


In [4]:
def name_components(scores):
    names = [f"PC{i}" for i in range(1, scores.shape[1] + 1)]
    return scores.set_axis(names, axis="columns")


preprocess = Pipeline(
    [
        ("drop_all_zero", DropAllZeroColumns()),
        ("spearman", SpearmanFilter(threshold=0.9)),
        ("scale", StandardScaler().set_output(transform="pandas")),
        (
            "pca",
            PCA(n_components=0.99, svd_solver="full").set_output(transform="pandas"),
        ),
        ("name_components", FunctionTransformer(name_components)),
    ]
)

### Inspect the exploratory fit

Fit on all flows to inspect the feature removals and retained variance. `X_reduced` contains the components. `df_pca` joins them to the label and other metadata by row ID for analysis.


In [5]:
X_reduced = preprocess.fit_transform(X)
df_pca = pd.concat([X_reduced, metadata], axis=1)

In [6]:
columns_to_drop = preprocess.named_steps["drop_all_zero"].dropped_columns_
correlation_removals = preprocess.named_steps["spearman"].removals_
pca = preprocess.named_steps["pca"]
variance_retained = pca.explained_variance_ratio_.sum()

print(f"All-zero columns removed: {', '.join(columns_to_drop)}")
print(f"{pca.n_components_} components retain {variance_retained:.2%} of variance.")
correlation_removals

All-zero columns removed: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, FIN Flag Count, PSH Flag Count, ECE Flag Count, Fwd Avg Bytes/Bulk, Fwd Avg Packets/Bulk, Fwd Avg Bulk Rate, Bwd Avg Bytes/Bulk, Bwd Avg Packets/Bulk, Bwd Avg Bulk Rate
20 components retain 99.12% of variance.


,Removed feature,Retained feature,Absolute Spearman correlation
0,Fwd Packet Length Max,Total Length of Fwd Packets,0.904084
1,Fwd Packet Length Mean,Fwd Packet Length Min,0.987166
2,Bwd Packet Length Max,Total Length of Bwd Packets,0.999984
3,Bwd Packet Length Mean,Total Length of Bwd Packets,0.999872
4,Flow Bytes/s,Fwd Packet Length Min,0.902712
5,Flow Packets/s,Flow Duration,0.996758
6,Flow IAT Mean,Flow Duration,0.995619
7,Flow IAT Max,Flow Duration,0.998428
8,Fwd IAT Mean,Fwd IAT Total,0.998426
9,Fwd IAT Std,Total Fwd Packets,0.922406


## Section 2 – Supervised classification

### Stratified per-flow split

Set aside 20% of flows for testing, then set aside 20% of the remaining flows for validation. That gives about 64% training, 16% validation, and 20% testing overall. Both steps stratify by `label`, so each subset keeps roughly the same class proportions as the full dataset. For example, `ddos_ntp` makes up about 1.5% of all flows and should make up about 1.5% of each subset. The fixed random seed makes the split repeatable.

Use the original measurements in `X` for the split. When training a classifier, fit a fresh copy of the preprocessing pipeline on the training rows so that feature filtering, scaling, and PCA learn from those rows alone. Use validation to compare model choices and save the test set for the final evaluation.


In [7]:
random_seed = 42
test_fraction = 0.20
validation_fraction_of_remaining = 0.20

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=test_fraction, stratify=y, random_state=random_seed
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=validation_fraction_of_remaining,
    stratify=y_train_val,
    random_state=random_seed,
)

In [8]:
split_counts = pd.DataFrame(
    {
        "All flows": y.value_counts(),
        "Train": y_train.value_counts(),
        "Validation": y_val.value_counts(),
        "Test": y_test.value_counts(),
    }
).sort_index()
split_counts.loc["Total"] = split_counts.sum()
split_counts

,All flows,Train,Validation,Test
label,,,,
benign,5658,3621,905,1132
ddos_dns,5369,3436,859,1074
ddos_ldap,5928,3794,948,1186
ddos_mssql,5911,3783,946,1182
ddos_netbios,5830,3731,933,1166
ddos_ntp,986,631,158,197
ddos_snmp,5984,3829,958,1197
ddos_ssdp,5970,3821,955,1194
ddos_syn,5480,3507,877,1096


### Random forest baseline

Use `clone(preprocess)` as the first step of a new pipeline. The clone copies the preprocessing settings and starts unfitted. When we call `fit` on `X_train`, the filters, scaler, and PCA learn from the training flows. For example, each validation flow is scaled using the training means and standard deviations. The random forest uses its default settings with a fixed seed.

This baseline uses the PCA features from Section 1. A random forest can also learn directly from the measurements. PCA reduces the number of inputs, but each component mixes measurements, which makes individual tree splits harder to interpret.

Compare training and validation accuracy and macro F1. Macro F1 gives each label equal weight, including `ddos_ntp`, which has fewer flows. Save the test set for final evaluation.


In [9]:
random_forest = Pipeline(
    [
        ("preprocess", clone(preprocess)),
        ("classifier", RandomForestClassifier(random_state=random_seed)),
    ]
)
random_forest.fit(X_train, y_train)

y_train_pred = random_forest.predict(X_train)
y_val_pred = random_forest.predict(X_val)

In [10]:
scores = pd.DataFrame(
    {
        "Accuracy": [
            accuracy_score(y_train, y_train_pred),
            accuracy_score(y_val, y_val_pred),
        ],
        "Macro F1": [
            f1_score(y_train, y_train_pred, average="macro"),
            f1_score(y_val, y_val_pred, average="macro"),
        ],
    },
    index=["Train", "Validation"],
)
scores.round(4)

,Accuracy,Macro F1
Train,0.7868,0.7859
Validation,0.7300,0.7245


In [11]:
print(classification_report(y_val, y_val_pred, digits=3, zero_division=0))

class_labels = random_forest.classes_
validation_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_val, y_val_pred, labels=class_labels),
    index=pd.Index(class_labels, name="Actual label"),
    columns=pd.Index(class_labels, name="Predicted label"),
)
validation_confusion_matrix

              precision    recall  f1-score   support

      benign      0.988     0.998     0.993       905
    ddos_dns      0.977     0.845     0.906       859
   ddos_ldap      0.489     0.640     0.555       948
  ddos_mssql      0.507     0.119     0.193       946
ddos_netbios      0.974     0.959     0.967       933
    ddos_ntp      0.867     0.905     0.885       158
   ddos_snmp      0.997     1.000     0.998       958
   ddos_ssdp      0.608     0.951     0.742       955
    ddos_syn      0.963     0.442     0.606       877
   ddos_tftp      0.628     0.982     0.766       842
    ddos_udp      0.532     0.653     0.586       940
ddos_udp_lag      0.566     0.441     0.495       958

    accuracy                          0.730     10279
   macro avg      0.758     0.745     0.724     10279
weighted avg      0.746     0.730     0.710     10279



Predicted label,benign,ddos_dns,ddos_ldap,ddos_mssql,ddos_netbios,ddos_ntp,ddos_snmp,ddos_ssdp,ddos_syn,ddos_tftp,ddos_udp,ddos_udp_lag
Actual label,,,,,,,,,,,,
benign,903,1,0,0,0,1,0,0,0,0,0,0
ddos_dns,3,726,21,1,11,21,3,72,0,0,1,0
ddos_ldap,0,0,607,85,3,0,0,253,0,0,0,0
ddos_mssql,1,1,583,113,1,0,0,247,0,0,0,0
ddos_netbios,0,4,12,3,895,0,0,13,0,0,5,1
ddos_ntp,7,8,0,0,0,143,0,0,0,0,0,0
ddos_snmp,0,0,0,0,0,0,958,0,0,0,0,0
ddos_ssdp,0,3,18,21,5,0,0,908,0,0,0,0
ddos_syn,0,0,0,0,0,0,0,0,388,489,0,0


### Tune the number of trees with cross-validation

Use `classifier__n_estimators` to try 100, 200, and 300 trees on the pipeline step named `classifier`. Split `X_train` into five stratified folds so each fold has roughly the same label proportions. For example, the 200-tree pipeline trains on four folds and scores on the fifth, repeating until every fold has been scored once. The filters, scaler, and PCA are fitted again on the training rows of each fold. The separate validation and test sets are reserved for evaluation.

Macro F1 gives every label equal weight, so a rare label such as `ddos_ntp` counts as much as a common one. The search selects the tree count with the highest mean macro F1 and refits that pipeline on all of `X_train`. The table shows the mean score, its standard deviation across folds, and the mean time to fit the full pipeline per fold.

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
forest_search = GridSearchCV(
    random_forest,
    param_grid={"classifier__n_estimators": [100, 200, 300]},
    scoring="f1_macro",
    cv=cv,
)
forest_search.fit(X_train, y_train)

cv_columns = {
    "param_classifier__n_estimators": "Number of trees",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
forest_cv_results = pd.DataFrame(forest_search.cv_results_)[list(cv_columns)].rename(
    columns=cv_columns
)

best_tree_count = forest_search.best_params_["classifier__n_estimators"]
print(f"Selected number of trees: {best_tree_count}")
forest_cv_results.round(4)

Selected number of trees: 300


,Number of trees,Mean CV macro F1,CV macro F1 std,Mean fit time (s)
0,100,0.7256,0.0047,2.6317
1,200,0.7263,0.0046,5.0135
2,300,0.7264,0.0043,7.5216


### Repeat the search without PCA

Clone the search to reuse the same five stratified folds, tree counts, and macro F1 scoring. Set the PCA and component naming steps to `passthrough` so the forest sees the filtered, standardized measurements. For example, the 200-tree candidate uses the same training and validation rows in both searches. The all-zero filter, Spearman filter, and scaler still learn from the training rows of each fold.

In [13]:
forest_search_without_pca = clone(forest_search).set_params(
    estimator__preprocess__pca="passthrough",
    estimator__preprocess__name_components="passthrough",
)
forest_search_without_pca.fit(X_train, y_train)

forest_cv_results_without_pca = pd.DataFrame(
    forest_search_without_pca.cv_results_, columns=list(cv_columns)
).rename(columns=cv_columns)

best_tree_count_without_pca = forest_search_without_pca.best_params_[
    "classifier__n_estimators"
]
print(f"Selected number of trees without PCA: {best_tree_count_without_pca}")
forest_cv_results_without_pca.round(4)

Selected number of trees without PCA: 300


,Number of trees,Mean CV macro F1,CV macro F1 std,Mean fit time (s)
0,100,0.7272,0.0041,1.0345
1,200,0.7269,0.0051,1.9301
2,300,0.7274,0.0045,2.8266


### Tune the PCA variance threshold

Fix the random forest at 100 trees, the best tree count in both earlier searches. Compare PCA thresholds of two-thirds, 80%, 90%, and 95% using the same five stratified folds and macro F1 scoring. Each fold fits its own filters, scaler, and PCA using its training rows. For example, at 80%, PCA keeps enough components to explain at least 80% of the variance in that fold's filtered and standardized training data.

In [14]:
pca_variance_search = clone(forest_search).set_params(
    estimator__classifier__n_estimators=100,
    param_grid={"preprocess__pca__n_components": [2 / 3, 0.80, 0.90, 0.95]},
)
pca_variance_search.fit(X_train, y_train)

pca_cv_columns = {
    "param_preprocess__pca__n_components": "PCA variance threshold",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
pca_cv_results = pd.DataFrame(
    pca_variance_search.cv_results_, columns=list(pca_cv_columns)
).rename(columns=pca_cv_columns)
pca_cv_results["PCA variance threshold"] = pca_cv_results["PCA variance threshold"].map(
    "{:.2%}".format
)

best_variance_threshold = pca_variance_search.best_params_[
    "preprocess__pca__n_components"
]
best_preprocess = pca_variance_search.best_estimator_.named_steps["preprocess"]
best_pca = best_preprocess.named_steps["pca"]
print(f"Selected PCA variance threshold: {best_variance_threshold:.2%}")
print(f"Components kept after refitting on all training rows: {best_pca.n_components_}")
pca_cv_results.round(4)

Selected PCA variance threshold: 95.00%
Components kept after refitting on all training rows: 16


,PCA variance threshold,Mean CV macro F1,CV macro F1 std,Mean fit time (s)
0,66.67%,0.7251,0.0060,1.5903
1,80.00%,0.7247,0.0061,2.0993
2,90.00%,0.7254,0.0050,2.1064
3,95.00%,0.7254,0.0052,2.6436


## Logistic regression

In [15]:
lr = Pipeline([
    ("preprocess", clone(preprocess)),
    ("classifier", LogisticRegression(solver="lbfgs", max_iter=1000)),
])

lr_search = GridSearchCV(
    lr,
    param_grid={
        "classifier__C": [0.001, 0.01, 0.1, 1, 10],
        "classifier__class_weight": [None, "balanced"],
        "preprocess__pca__n_components": [0.50, 0.66, 0.80, 0.90],
    },
    scoring="f1_macro",
    cv=cv,
)
lr_search.fit(X_train, y_train)
print(f"Best mean CV macro F1: {lr_search.best_score_:.4f}")
lr_search.best_params_

/Users/lents/coding/polito/projects/ml/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/lents/coding/polito/projects/ml/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https:/

Best mean CV macro F1: 0.6120


{'classifier__C': 10,
 'classifier__class_weight': 'balanced',
 'preprocess__pca__n_components': 0.9}

In [16]:
lr_cv_columns = {
    "rank_test_score": "Rank",
    "param_preprocess__pca__n_components": "PCA variance threshold",
    "param_classifier__C": "C",
    "param_classifier__class_weight": "Class weight",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
lr_cv_results = (
    pd.DataFrame(lr_search.cv_results_, columns=list(lr_cv_columns))
    .rename(columns=lr_cv_columns)
    .sort_values("Rank")
    .reset_index(drop=True)
)
lr_cv_results["Class weight"] = lr_cv_results["Class weight"].fillna("Uniform")
display(
    lr_cv_results.round({
        "Mean CV macro F1": 4,
        "CV macro F1 std": 4,
        "Mean fit time (s)": 2,
    }).assign(**{
        "PCA variance threshold": lr_cv_results["PCA variance threshold"].map("{:.0%}".format),
    }).set_index("Rank")
)

lr_val_pred = lr_search.predict(X_val)
print("Validation classification report")
print(classification_report(y_val, lr_val_pred, digits=3, zero_division=0))

,PCA variance threshold,C,Class weight,Mean CV macro F1,CV macro F1 std,Mean fit time (s)
Rank,,,,,,
1,90%,10.000,balanced,0.6120,0.0025,2.23
2,90%,10.000,Uniform,0.6015,0.0069,1.67
3,80%,10.000,balanced,0.5677,0.0107,2.10
4,90%,1.000,Uniform,0.5653,0.0102,0.83
5,90%,1.000,balanced,0.5652,0.0062,1.02
6,80%,10.000,Uniform,0.5642,0.0138,1.66
7,80%,1.000,balanced,0.5622,0.0056,0.98
8,66%,10.000,balanced,0.5612,0.0085,2.26
9,90%,0.100,balanced,0.5569,0.0050,0.48


Validation classification report
              precision    recall  f1-score   support

      benign      0.988     0.748     0.852       905
    ddos_dns      0.997     0.801     0.888       859
   ddos_ldap      0.472     0.071     0.123       948
  ddos_mssql      0.350     0.905     0.505       946
ddos_netbios      0.732     0.950     0.826       933
    ddos_ntp      0.381     0.930     0.540       158
   ddos_snmp      0.989     0.998     0.993       958
   ddos_ssdp      0.622     0.255     0.362       955
    ddos_syn      0.979     0.422     0.590       877
   ddos_tftp      0.620     0.999     0.765       842
    ddos_udp      0.514     0.530     0.522       940
ddos_udp_lag      0.522     0.358     0.425       958

    accuracy                          0.639     10279
   macro avg      0.681     0.664     0.616     10279
weighted avg      0.698     0.639     0.617     10279



## Naive Bayes

In [17]:
gnb = Pipeline([
    ("preprocess", clone(preprocess)),
    ("classifier", GaussianNB()),
])

gnb_search = GridSearchCV(
    gnb,
    param_grid={
        "classifier__var_smoothing": [1e-11, 1e-9, 1e-7, 1e-5, 1e-3],
        "preprocess__pca__n_components": [0.50, 0.66, 0.80, 0.90],
    },
    scoring="f1_macro",
    cv=cv,
)
gnb_search.fit(X_train, y_train)
print(f"Best mean CV macro F1: {gnb_search.best_score_:.4f}")
gnb_search.best_params_

Best mean CV macro F1: 0.5231


{'classifier__var_smoothing': 1e-07, 'preprocess__pca__n_components': 0.9}

In [18]:
gnb_cv_columns = {
    "rank_test_score": "Rank",
    "param_preprocess__pca__n_components": "PCA variance threshold",
    "param_classifier__var_smoothing": "Variance smoothing",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
gnb_cv_results = (
    pd.DataFrame(gnb_search.cv_results_, columns=list(gnb_cv_columns))
    .rename(columns=gnb_cv_columns)
    .sort_values("Rank")
    .reset_index(drop=True)
)
display(
    gnb_cv_results.round({
        "Mean CV macro F1": 4,
        "CV macro F1 std": 4,
        "Mean fit time (s)": 2,
    }).assign(**{
        "Variance smoothing": gnb_cv_results["Variance smoothing"].map("{:.0e}".format),
        "PCA variance threshold": gnb_cv_results["PCA variance threshold"].map("{:.0%}".format),
    }).set_index("Rank")
)

gnb_val_pred = gnb_search.predict(X_val)
print("Validation classification report")
print(classification_report(y_val, gnb_val_pred, digits=3, zero_division=0))

,PCA variance threshold,Variance smoothing,Mean CV macro F1,CV macro F1 std,Mean fit time (s)
Rank,,,,,
1,90%,1e-07,0.5231,0.0162,0.15
2,90%,1e-11,0.5229,0.0161,0.15
2,90%,1e-09,0.5229,0.0161,0.15
4,90%,1e-05,0.5200,0.0144,0.16
5,80%,1e-11,0.5109,0.0149,0.16
6,80%,1e-09,0.5108,0.0148,0.16
7,80%,1e-07,0.5106,0.0147,0.15
8,80%,1e-05,0.5068,0.0121,0.15
9,66%,1e-07,0.4854,0.0022,0.15


Validation classification report
              precision    recall  f1-score   support

      benign      0.888     0.702     0.784       905
    ddos_dns      0.970     0.786     0.868       859
   ddos_ldap      0.000     0.000     0.000       948
  ddos_mssql      0.325     0.986     0.489       946
ddos_netbios      0.939     0.426     0.586       933
    ddos_ntp      0.303     0.665     0.416       158
   ddos_snmp      0.998     0.979     0.988       958
   ddos_ssdp      0.579     0.076     0.135       955
    ddos_syn      0.895     0.380     0.533       877
   ddos_tftp      0.623     0.999     0.767       842
    ddos_udp      0.300     0.013     0.024       940
ddos_udp_lag      0.390     0.977     0.558       958

    accuracy                          0.572     10279
   macro avg      0.601     0.582     0.512     10279
weighted avg      0.617     0.572     0.513     10279



## One-vs-rest classification

Each flow has one label, so pass the original one-dimensional `y_train` to `OneVsRestClassifier`. It creates one binary target per class internally. For example, the `ddos_dns` model sees `1` for DNS attack flows and `0` for every other flow. Passing a two-dimensional indicator matrix would make scikit-learn treat the task as multilabel, where a flow can receive several labels.

LASSO fits the 0/1 targets as a regression problem. The wrapper uses each regressor's prediction as a score and returns the label with the highest score. Keep preprocessing before the wrapper so each fold fits one scaler and one PCA on its training rows and shares the resulting components across all class models. Start with 90% retained variance to match the selected PCA setting in the previous two searches.

For another binary estimator, clone `lasso_ovr` and replace `classifier__estimator` with a classifier that exposes `decision_function` or `predict_proba`. Use `classifier__estimator__alpha` to tune LASSO's regularization.

In [19]:
lasso_ovr = Pipeline([
    ("preprocess", clone(preprocess).set_params(pca__n_components=0.90)),
    ("classifier", OneVsRestClassifier(Lasso(alpha=1e-3))),
])

In [20]:
lasso_ovr_search = GridSearchCV(
    lasso_ovr,
    param_grid={"classifier__estimator__alpha": [0.5, 1, 1.5, 2, 3]},
    scoring="f1_macro",
    cv=cv,
)
lasso_ovr_search.fit(X_train, y_train)
print(f"Best mean CV macro F1: {lasso_ovr_search.best_score_:.4f}")
lasso_ovr_search.best_params_

Best mean CV macro F1: 0.0142


{'classifier__estimator__alpha': 0.5}

In [21]:
lasso_ovr_cv_columns = {
    "rank_test_score": "Rank",
    "param_classifier__estimator__alpha": "Alpha",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
lasso_ovr_cv_results = (
    pd.DataFrame(lasso_ovr_search.cv_results_, columns=list(lasso_ovr_cv_columns))
    .rename(columns=lasso_ovr_cv_columns)
    .sort_values("Rank", kind="stable")
    .set_index("Rank")
)
display(lasso_ovr_cv_results.round({
    "Mean CV macro F1": 4,
    "CV macro F1 std": 4,
    "Mean fit time (s)": 2,
}))

lasso_ovr_val_pred = lasso_ovr_search.predict(X_val)
print("Validation classification report")
print(classification_report(y_val, lasso_ovr_val_pred, digits=3, zero_division=0))

,Alpha,Mean CV macro F1,CV macro F1 std,Mean fit time (s)
Rank,,,,
1,0.5,0.0142,0.0,0.19
1,1.0,0.0142,0.0,0.19
1,1.5,0.0142,0.0,0.19
1,2.0,0.0142,0.0,0.19
1,3.0,0.0142,0.0,0.19


Validation classification report
              precision    recall  f1-score   support

      benign      0.000     0.000     0.000       905
    ddos_dns      0.000     0.000     0.000       859
   ddos_ldap      0.000     0.000     0.000       948
  ddos_mssql      0.000     0.000     0.000       946
ddos_netbios      0.000     0.000     0.000       933
    ddos_ntp      0.000     0.000     0.000       158
   ddos_snmp      0.000     0.000     0.000       958
   ddos_ssdp      0.000     0.000     0.000       955
    ddos_syn      0.000     0.000     0.000       877
   ddos_tftp      0.000     0.000     0.000       842
    ddos_udp      0.000     0.000     0.000       940
ddos_udp_lag      0.093     1.000     0.171       958

    accuracy                          0.093     10279
   macro avg      0.008     0.083     0.014     10279
weighted avg      0.009     0.093     0.016     10279



## Support vector classifier

Use the RBF kernel with `gamma="scale"` and retain 90% of the variance with PCA. Compare three values of `C` with uniform and balanced class weights using the same stratified folds and macro F1 scoring.

In [22]:
svc = Pipeline([
    ("preprocess", clone(preprocess).set_params(pca__n_components=0.90)),
    ("classifier", SVC(kernel="rbf", gamma="scale")),
])

svc_search = GridSearchCV(
    svc,
    param_grid={
        "classifier__C": [0.1, 1, 10],
        "classifier__class_weight": [None, "balanced"],
    },
    scoring="f1_macro",
    cv=cv,
)
svc_search.fit(X_train, y_train)
print(f"Best mean CV macro F1: {svc_search.best_score_:.4f}")
svc_search.best_params_

Best mean CV macro F1: 0.6376


{'classifier__C': 10, 'classifier__class_weight': 'balanced'}

In [23]:
svc_cv_columns = {
    "rank_test_score": "Rank",
    "param_classifier__C": "C",
    "param_classifier__class_weight": "Class weight",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
svc_cv_results = (
    pd.DataFrame(svc_search.cv_results_, columns=list(svc_cv_columns))
    .rename(columns=svc_cv_columns)
    .sort_values("Rank")
    .reset_index(drop=True)
)
svc_cv_results["Class weight"] = svc_cv_results["Class weight"].fillna("Uniform")
display(
    svc_cv_results.round({
        "Mean CV macro F1": 4,
        "CV macro F1 std": 4,
        "Mean fit time (s)": 2,
    }).set_index("Rank")
)

svc_val_pred = svc_search.predict(X_val)
print("Validation classification report")
print(classification_report(y_val, svc_val_pred, digits=3, zero_division=0))

,C,Class weight,Mean CV macro F1,CV macro F1 std,Mean fit time (s)
Rank,,,,,
1,10.0,balanced,0.6376,0.0021,3.05
2,10.0,Uniform,0.6342,0.0100,2.77
3,1.0,balanced,0.5834,0.0167,3.64
4,1.0,Uniform,0.5655,0.0115,3.26
5,0.1,balanced,0.5144,0.0022,4.95
6,0.1,Uniform,0.4999,0.0016,4.42


Validation classification report
              precision    recall  f1-score   support

      benign      0.989     0.874     0.928       905
    ddos_dns      0.999     0.786     0.879       859
   ddos_ldap      0.468     0.070     0.121       948
  ddos_mssql      0.346     0.942     0.507       946
ddos_netbios      0.939     0.847     0.891       933
    ddos_ntp      0.530     0.962     0.683       158
   ddos_snmp      0.991     0.998     0.994       958
   ddos_ssdp      0.751     0.221     0.341       955
    ddos_syn      1.000     0.424     0.596       877
   ddos_tftp      0.625     1.000     0.769       842
    ddos_udp      0.514     0.530     0.522       940
ddos_udp_lag      0.472     0.506     0.488       958

    accuracy                          0.655     10279
   macro avg      0.719     0.680     0.643     10279
weighted avg      0.729     0.655     0.636     10279



## K-nearest neighbors

Use Euclidean distance on the PCA components that retain 90% of the variance. Compare six neighborhood sizes and two voting rules. For example, with five neighbors, uniform voting gives each neighbor one vote, while distance weighting gives closer neighbors more influence.

In [25]:
knn = Pipeline([
    ("preprocess", clone(preprocess).set_params(pca__n_components=0.90)),
    ("classifier", KNeighborsClassifier()),
])

knn_search = GridSearchCV(
    knn,
    param_grid={
        "classifier__n_neighbors": [3, 5, 11, 13, 15, 20],
        "classifier__weights": ["uniform", "distance"],
    },
    scoring="f1_macro",
    cv=cv,
)
knn_search.fit(X_train, y_train)
print(f"Best mean CV macro F1: {knn_search.best_score_:.4f}")
knn_search.best_params_

Best mean CV macro F1: 0.7473


{'classifier__n_neighbors': 15, 'classifier__weights': 'distance'}

In [26]:
knn_cv_columns = {
    "rank_test_score": "Rank",
    "param_classifier__n_neighbors": "Neighbors",
    "param_classifier__weights": "Voting weights",
    "mean_test_score": "Mean CV macro F1",
    "std_test_score": "CV macro F1 std",
    "mean_fit_time": "Mean fit time (s)",
}
knn_cv_results = (
    pd.DataFrame(knn_search.cv_results_, columns=list(knn_cv_columns))
    .rename(columns=knn_cv_columns)
    .sort_values("Rank")
    .reset_index(drop=True)
)
display(
    knn_cv_results.round({
        "Mean CV macro F1": 4,
        "CV macro F1 std": 4,
        "Mean fit time (s)": 2,
    }).set_index("Rank")
)

knn_val_pred = knn_search.predict(X_val)
print("Validation classification report")
print(classification_report(y_val, knn_val_pred, digits=3, zero_division=0))

,Neighbors,Voting weights,Mean CV macro F1,CV macro F1 std,Mean fit time (s)
Rank,,,,,
1,15,distance,0.7473,0.0065,0.18
2,13,distance,0.7446,0.0058,0.27
3,20,distance,0.7439,0.0068,0.16
4,11,distance,0.7436,0.0090,0.27
5,15,uniform,0.7401,0.0067,0.27
6,11,uniform,0.7384,0.0094,0.27
7,13,uniform,0.7379,0.0046,0.26
8,20,uniform,0.7351,0.0075,0.16
9,5,distance,0.7303,0.0116,0.27


Validation classification report
              precision    recall  f1-score   support

      benign      0.990     0.993     0.992       905
    ddos_dns      0.958     0.858     0.905       859
   ddos_ldap      0.543     0.658     0.595       948
  ddos_mssql      0.585     0.415     0.486       946
ddos_netbios      0.976     0.966     0.971       933
    ddos_ntp      0.851     0.937     0.892       158
   ddos_snmp      0.996     1.000     0.998       958
   ddos_ssdp      0.733     0.848     0.786       955
    ddos_syn      0.586     0.877     0.703       877
   ddos_tftp      0.737     0.359     0.482       842
    ddos_udp      0.539     0.465     0.499       940
ddos_udp_lag      0.538     0.609     0.571       958

    accuracy                          0.736     10279
   macro avg      0.753     0.749     0.740     10279
weighted avg      0.744     0.736     0.729     10279

